# Day 2. From a model to a measurement

**Instructor notebook.** Runs on paid Colab Pro with GPU.

We pick up where Day 1 left off. Same NTE corpus. Same off-the-shelf DeBERTa. The new things are a hand-coded gold set, a thirteen-hypothesis ensemble, and a country-year proxy plotted against known events.

**Before you run.** Make sure `NTE_IPR_final_v2.csv` is uploaded to your Google Drive at `MyDrive/nte_text/NTE_IPR_final_v2.csv`.

## 1. Setup

Mount Drive, install libraries, set up imports.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers scikit-learn

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import pipeline
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from scipy.stats import pearsonr

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 2. Load the NTE corpus

Same CSV as Day 1. Four columns. `country` (ISO3), `year`, `text` (the IP-section paragraph), and `deberta_score` (the fine-tuned score from Park's published proxy, scaled to roughly &minus;5 to +5).

We will treat `deberta_score` as a stand-in for a hand-coded gold set today. In a real project this column would be replaced by hand-codes from you and a co-author. The validation logic is identical.

In [ ]:
CSV_PATH = '/content/drive/MyDrive/nte_text/NTE_IPR_final_v2.csv'
df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape}')
print(f'Years: {df.year.min()} to {df.year.max()}')
print(f'Countries: {df.country.nunique()}')
df.head(3)

## 3. Build the 30-paragraph gold set

Stratify by year so the gold set covers the full 1995-2022 span. Binarize `deberta_score` into three classes (&minus;1 / 0 / +1) to mirror what a human coder would produce.

Thresholds are deliberate. Anything above +1 on the &minus;5 to +5 scale is praise. Anything below &minus;1 is criticism. The rest is mixed.

In [ ]:
def to_class(s, hi=1.0, lo=-1.0):
    if s > hi: return 1
    if s < lo: return -1
    return 0

df['gold'] = df['deberta_score'].apply(to_class)
print('Gold distribution on full corpus:')
print(df['gold'].value_counts().sort_index())

rng = np.random.default_rng(42)
year_bins = pd.cut(df['year'], bins=6, labels=False)
gold_idx = []
for b in sorted(year_bins.unique()):
    pool = df.index[year_bins == b].tolist()
    gold_idx.extend(rng.choice(pool, size=5, replace=False).tolist())

gold = df.loc[gold_idx].reset_index(drop=True)
print(f'\nGold set: {len(gold)} paragraphs, {gold.country.nunique()} countries, years {gold.year.min()}-{gold.year.max()}')
print('Gold class balance:', gold['gold'].value_counts().sort_index().to_dict())

## 4. Load the encoder

Same model as Day 1. `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`. ~184M params, MIT license.

In [ ]:
MODEL_ID = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
device = 0 if torch.cuda.is_available() else -1
zsc = pipeline('zero-shot-classification', model=MODEL_ID, device=device)
print(f'Loaded {MODEL_ID} on {"GPU" if device == 0 else "CPU"}')

## 5. Baseline. One hypothesis, three labels

Day 1's setup. Three candidate labels, multi-class softmax across them. The predicted class is the argmax.

We score each gold-set paragraph and compare predictions against the gold.

In [ ]:
baseline_labels = [
    'the country has weak IP protection and the US criticizes it',
    'the country has strong IP protection and the US praises it',
    'the paragraph is unrelated or balanced',
]
label_to_class = {
    baseline_labels[0]: -1,
    baseline_labels[1]: 1,
    baseline_labels[2]: 0,
}

t0 = time.time()
baseline_preds = zsc(gold['text'].tolist(), candidate_labels=baseline_labels, batch_size=8)
print(f'Baseline ran in {time.time()-t0:.1f} sec')

gold['pred_baseline'] = [label_to_class[p['labels'][0]] for p in baseline_preds]
kappa_base = cohen_kappa_score(gold['gold'], gold['pred_baseline'])
print(f'\nBaseline kappa vs gold: {kappa_base:.3f}')
print('\nConfusion matrix (rows = gold, cols = predicted):')
print(pd.DataFrame(
    confusion_matrix(gold['gold'], gold['pred_baseline'], labels=[-1,0,1]),
    index=['gold=-1','gold= 0','gold=+1'],
    columns=['pred=-1','pred= 0','pred=+1'],
))

## 6. The 13-hypothesis ensemble

Each hypothesis is checked independently against the paragraph. We use `multi_label=True` so each hypothesis returns its own probability rather than competing with the others on a softmax. The final score is a weighted average. Positive weights pull toward +1. Negative weights pull toward &minus;1.

Weights below come from the codebook. They are not learned. A real research project would set them based on the codebook author's judgment, then justify them in the appendix.

In [ ]:
hypotheses = [
    ('The text praises the country IP enforcement.', +1.0),
    ('The text describes the country making progress on IP protection.', +1.0),
    ('The text mentions the country acceding to a WIPO treaty.', +0.7),
    ('The text mentions the country being removed from the Special 301 Report.', +0.7),
    ('The text reflects strong cooperation between the US and the country on IP.', +0.5),
    ('The text describes effective enforcement of trademarks or copyright in the country.', +0.5),
    ('The text criticizes the country IP enforcement.', -1.0),
    ('The text reports widespread piracy in the country.', -1.0),
    ('The text describes the country being placed on the Watch List or Priority Watch List.', -0.8),
    ('The text expresses US concern about counterfeiting in the country.', -0.7),
    ('The text reports law enforcement gaps on IP in the country.', -0.6),
    ('The text reports inadequate pharmaceutical patent protection in the country.', -0.6),
    ('The text reports stalled or weak IP-related reform in the country.', -0.6),
]
hyp_texts = [h for h, _ in hypotheses]
hyp_weights = np.array([w for _, w in hypotheses])
print(f'{len(hypotheses)} hypotheses. Sum of positive weights = {hyp_weights[hyp_weights>0].sum():.1f}. Sum of negative weights = {hyp_weights[hyp_weights<0].sum():.1f}.')

In [ ]:
def ensemble_scores(texts, hyp_texts, hyp_weights, batch_size=8):
    preds = zsc(texts, candidate_labels=hyp_texts, multi_label=True, batch_size=batch_size)
    out = []
    weight_norm = np.abs(hyp_weights).sum()
    for p in preds:
        prob_by_text = dict(zip(p['labels'], p['scores']))
        probs = np.array([prob_by_text[h] for h in hyp_texts])
        score = float((probs * hyp_weights).sum() / weight_norm)
        out.append(score)
    return out

t0 = time.time()
gold['score_ensemble'] = ensemble_scores(gold['text'].tolist(), hyp_texts, hyp_weights)
print(f'Ensemble on 30 paragraphs ran in {time.time()-t0:.1f} sec')

def score_to_class(s, hi=0.04, lo=-0.04):
    if s > hi: return 1
    if s < lo: return -1
    return 0

gold['pred_ensemble'] = gold['score_ensemble'].apply(score_to_class)

## 7. Compare baseline vs ensemble against the gold set

The two metrics that belong in the appendix are kappa and Pearson correlation. Accuracy alone is unreliable because the class distribution is imbalanced.

In [ ]:
kappa_ens = cohen_kappa_score(gold['gold'], gold['pred_ensemble'])
r_ens, _ = pearsonr(gold['score_ensemble'], gold['deberta_score'])

print(f'{"approach":<28} {"kappa":>8} {"corr":>8} {"acc":>8}')
print('-' * 56)
print(f'{"baseline (1 hypothesis)":<28} {kappa_base:>8.3f} {"-":>8} {(gold["gold"]==gold["pred_baseline"]).mean():>8.3f}')
print(f'{"ensemble (13 hypotheses)":<28} {kappa_ens:>8.3f} {r_ens:>8.3f} {(gold["gold"]==gold["pred_ensemble"]).mean():>8.3f}')

print('\nEnsemble confusion matrix (rows = gold, cols = predicted):')
print(pd.DataFrame(
    confusion_matrix(gold['gold'], gold['pred_ensemble'], labels=[-1,0,1]),
    index=['gold=-1','gold= 0','gold=+1'],
    columns=['pred=-1','pred= 0','pred=+1'],
))

## 8. Look at the disagreements

Confusion-matrix off-diagonals are the cases worth re-reading by hand. Sometimes the gold is wrong and the model is right. Sometimes the codebook needs a new rule. Sometimes the paragraph really is mixed and the model picked one side.

In [ ]:
misses = gold[gold['gold'] != gold['pred_ensemble']].copy()
print(f'{len(misses)} disagreements out of {len(gold)}')
for _, r in misses.head(5).iterrows():
    print(f'\n[{r.country} {r.year}] gold={r.gold:+d}  pred={r.pred_ensemble:+d}  score={r.score_ensemble:+.2f}')
    print('  ' + r.text[:280] + ('...' if len(r.text) > 280 else ''))

## 9. Run the ensemble on a larger sample

Now that we have a defensible per-paragraph score, we can scale up. To keep the livestream moving we use a 600-paragraph stratified sample. In a real project you would run the ensemble on every paragraph in the corpus.

In [ ]:
sample = df.sample(n=600, random_state=0).reset_index(drop=True)

t0 = time.time()
sample['score_ensemble'] = ensemble_scores(sample['text'].tolist(), hyp_texts, hyp_weights)
print(f'Ensemble on {len(sample)} paragraphs ran in {time.time()-t0:.1f} sec')

r_full, _ = pearsonr(sample['score_ensemble'], sample['deberta_score'])
print(f'Correlation against fine-tuned proxy on the 600-paragraph sample: {r_full:+.3f}')

## 10. Aggregate to country-year

The simplest aggregation. Group by `country` and `year`, average the paragraph scores. We also keep the count so the appendix can flag thinly-sampled country-years.

In [ ]:
cy = (sample
      .groupby(['country','year'])
      .agg(score=('score_ensemble','mean'),
           n_paragraphs=('score_ensemble','size'))
      .reset_index())
print(f'{len(cy)} country-year observations')
print(cy.head(8).to_string(index=False))

## 11. Plot two cases against known events

Israel was removed from the Special 301 Report in 2014. Turkey faced a coup attempt in mid-2016 followed by sustained democratic erosion. Both events are external. Neither is hand-engineered into the model. If the proxy is doing its job, both should show up in the time series.

In [ ]:
cases = ['ISR', 'TUR']
fig, ax = plt.subplots(figsize=(10, 4.5))
for code in cases:
    sub = cy[cy['country'] == code].sort_values('year')
    if len(sub) == 0:
        print(f'No rows for {code} in the 600-paragraph sample. Try a larger sample.')
        continue
    ax.plot(sub['year'], sub['score'], marker='o', label=code, linewidth=2)

ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.axvline(2014, color='#B1F1B1', linewidth=2, linestyle='--', alpha=0.8)
ax.axvline(2016, color='#B1F1B1', linewidth=2, linestyle='--', alpha=0.8)
ax.text(2014.1, ax.get_ylim()[1]*0.9, '2014\nIsrael removed', fontsize=9, color='#2D493C')
ax.text(2016.1, ax.get_ylim()[1]*0.7, '2016\nTurkey coup attempt', fontsize=9, color='#2D493C')
ax.set_xlabel('Year')
ax.set_ylabel('Country-year proxy score')
ax.set_title('Ensemble proxy plotted against external events')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Save the proxy CSV

What the rest of the paper plugs into. One row per country-year, plus the paragraph count for caveats.

In [ ]:
OUT_PATH = '/content/drive/MyDrive/nte_text/nte_proxy_country_year.csv'
cy.to_csv(OUT_PATH, index=False)
print(f'Wrote {len(cy)} rows to {OUT_PATH}')

## What we just built

1. A 30-paragraph gold set, stratified by year.
2. A baseline single-hypothesis classifier and a 13-hypothesis weighted ensemble.
3. Validation against gold via Cohen's kappa, Pearson correlation, and confusion matrix.
4. A country-year aggregate proxy plotted against two external events.
5. A CSV ready to merge into a regression dataset.

**Day 3** introduces generative models. Same reproducibility demands. Different toolkit. Useful for tasks the encoder cannot easily express as a label.